In [ ]:
# Optional: Export to ONNX for deployment
print("\nOptional: Export to ONNX format")
print("Uncomment below to export to ONNX:")
print("# model.export(format='onnx', imgsz=416, half=True)")

# Optional: Test the trained model
print("\nOptional: Test the trained model on an image")
print("Uncomment below to test:")
print("# test_image = 'path/to/test/image.jpg'")
print("# results = model(test_image)")
print("# results[0].show()")

## 7. Export Trained Weights

Save the trained model to the models directory for deployment.

In [ ]:
# pyright: reportUndefinedVariable=false
from pathlib import Path
import shutil

# Export trained model
export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

# Get the last trained model
best_model_path = results_path / 'weights' / 'best.pt'
last_model_path = results_path / 'weights' / 'last.pt'

if best_model_path.exists():
    # Copy best model
    destination = export_dir / 'yolov8-face-best.pt'
    shutil.copy(best_model_path, destination)
    print(f"✓ Best model saved to {destination}")

if last_model_path.exists():
    # Copy last model
    destination = export_dir / 'yolov8-face-last.pt'
    shutil.copy(last_model_path, destination)
    print(f"✓ Last model saved to {destination}")

# List exported models
print("\nExported Models:")
for model_file in export_dir.glob("*.pt"):
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  {model_file.name} ({size_mb:.1f} MB)")

In [ ]:
# pyright: reportUndefinedVariable=false
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Visualize training results
results_path = Path('runs/detect/face_detection')
if results_path.exists():
    # Plot training curves
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Load results CSV if it exists
    results_csv = results_path / 'results.csv'
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        
        if 'epoch' in df.columns:
            axes[0, 0].plot(df['epoch'], df['train/loss'])
            axes[0, 0].set_title('Training Loss')
            axes[0, 0].set_xlabel('Epoch')
            axes[0, 0].set_ylabel('Loss')
            axes[0, 0].grid(True)
    
    plt.tight_layout()
    plt.show()
    print("✓ Training results visualized")

In [ ]:
# pyright: reportUndefinedVariable=false

# Evaluate on validation set
print("Running validation...")
metrics = model.val(
    data=str(yaml_path),
    device=training_config['device'],
    verbose=True,
)

print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP75: {metrics.box.map75:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 6. Evaluate Performance

Validate the trained model and compute metrics.

In [ ]:
# pyright: reportUndefinedVariable=false

# Train the model
print("Starting training...")
print("=" * 50)

results = model.train(
    data=str(yaml_path),
    epochs=training_config['epochs'],
    imgsz=training_config['imgsz'],
    batch=training_config['batch'],
    patience=training_config['patience'],
    device=training_config['device'],
    optimizer=training_config['optimizer'],
    lr0=training_config['lr0'],
    lrf=training_config['lrf'],
    project='runs/detect',
    name='face_detection',
    exist_ok=False,
    verbose=True,
)

print("=" * 50)
print("✓ Training complete")

In [ ]:
# Configure training parameters
training_config = {
    'epochs': 100,
    'batch': 16,
    'imgsz': 416,
    'device': 0,  # GPU device (0 for first GPU, or 'cpu')
    'patience': 20,  # early stopping patience
    'save': True,
    'save_period': 10,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'optimizer': 'SGD',
    'lr0': 0.01,  # initial learning rate
    'lrf': 0.01,  # final learning rate
}

print("Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 5. Train the Model

Start the training process with configured hyperparameters.

In [ ]:
# Load YOLOv8 model
# Options: yolov8n.pt (nano), yolov8s.pt (small), yolov8m.pt (medium), yolov8l.pt (large)
model_name = "yolov8n"
model = YOLO(f"{model_name}.pt")

print(f"✓ Loaded {model_name} model")
print(f"  Model parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# Display model information
print(f"\nModel info:")
model.info()

## 4. Configure YOLOv8 Model

Load YOLOv8 model and configure hyperparameters for face detection training.

In [ ]:
# Create YAML configuration file for dataset
dataset_yaml = {
    'path': str(dataset_dir),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,  # number of classes (faces)
    'names': ['face']  # class names
}

yaml_path = dataset_dir / "data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f)

print(f"✓ Dataset YAML configuration created:")
print(f"  Path: {yaml_path}")
print(yaml.dump(dataset_yaml, default_flow_style=False))

In [ ]:
# Create dataset directory structure
dataset_dir = Path("datasets/face_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

(dataset_dir / "images" / "train").mkdir(parents=True, exist_ok=True)
(dataset_dir / "images" / "val").mkdir(parents=True, exist_ok=True)
(dataset_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)
(dataset_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)

print(f"✓ Dataset directory structure created at {dataset_dir}")

# Check existing dataset
train_images = list((dataset_dir / "images" / "train").glob("*"))
train_labels = list((dataset_dir / "labels" / "train").glob("*.txt"))
val_images = list((dataset_dir / "images" / "val").glob("*"))

print(f"\nDataset Status:")
print(f"  Training images: {len(train_images)}")
print(f"  Training labels: {len(train_labels)}")
print(f"  Validation images: {len(val_images)}")
print(f"\nNote: Upload your dataset to these directories before proceeding")

## 3. Prepare Dataset

Organize your training data and create the YAML configuration file.

In [ ]:
import torch
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import yaml
import matplotlib.pyplot as plt

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Import Required Libraries

Import Python libraries for YOLOv8 training and utility functions.

In [ ]:
# Mount Google Drive (if running on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except ImportError:
    print("⚠ Not running on Colab, skipping Drive mount")
    
import os
os.makedirs("datasets", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("✓ Local directories created")

In [ ]:
# Install and upgrade necessary packages
import subprocess
import sys

# Install YOLOv8
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "ultralytics"])

# Install additional dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python", "torch", "tensorboard"])

print("✓ Environment setup complete")

## 1. Setup Environment

Install required packages and configure the Colab environment for YOLOv8 training.

# YOLOv8 Face Detection Training

This notebook trains a YOLOv8 model for face detection in the context of the Sentinel AI project.
Designed for Google Colab environments with GPU acceleration.